# Task 1: Build a Model Comparison Function



> Required Imports

In [8]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import MultinomialNB

import re

> Load Dataset

In [9]:
train = pd.read_csv('reviews_train.csv')
test  = pd.read_csv('reviews_test.csv')

X_train = train['review']
y_train = train['label']
X_test  = test['review']
y_test  = test['label']

print(f"Train: {len(train)} rows | Test: {len(test)} rows")
train.head()

Train: 6666 rows | Test: 1000 rows


,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


> Clean the Dataset

In [10]:
def is_bad_data(review: str) -> bool:
    return bool(re.search(r"<[^>]+>", str(review)))

# Create the cleaned training set
train_clean = train[~train['review'].map(is_bad_data)]

# Define cleaned training variables
X_train_clean = train_clean['review']
y_train_clean = train_clean['label']

> Model Evaluation Function

In [11]:
def evaluate_lab_models(models, X_train, y_train, X_test, y_test):
    performance_list = []

    # Accept either a dictionary or a list of models
    if isinstance(models, dict):
        model_items = models.items()
    else:
        model_items = [(model.__class__.__name__, model) for model in models]

    for name, model in model_items:
        pipeline = Pipeline([
            ('vect', CountVectorizer()),
            ('tfidf', TfidfTransformer()),
            ('clf', model),
        ])

        # Train the model
        pipeline.fit(X_train, y_train)

        # Make predictions on the test set
        y_pred = pipeline.predict(X_test)

        # Evaluate performance
        performance_list.append({
            "Model name": name,
            "Accuracy": round(accuracy_score(y_test, y_pred), 4),
            "Precision": round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
            "Recall": round(recall_score(y_test, y_pred, average='weighted', zero_division=0), 4),
            "F1-score": round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        })

    return pd.DataFrame(performance_list).sort_values(by="F1-score", ascending=False).reset_index(drop=True)

> Model Definitions

In [12]:
def get_models():
    return {
        "Linear SVM":    LinearSVC(),
        "RBF SVM":       SVC(kernel="rbf", gamma="scale"),
        "KNN":           KNeighborsClassifier(n_neighbors=3),
        "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42),
        "Neural Net":    MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42),
        "AdaBoost":      AdaBoostClassifier(random_state=42),
        "Naive Bayes":   MultinomialNB(),
    }

> Model Training & Evaluation Execution

In [13]:
# 1. Initialize models
models_dict = get_models()

# 2. Evaluate on Original Data (Model-Centric attempt)
print("Evaluating models on ORIGINAL data...")
results_original = evaluate_lab_models(models_dict, X_train, y_train, X_test, y_test)

# 3. Evaluate on Cleaned Data (Data-Centric success)
print("Evaluating models on CLEANED data...")
results_cleaned = evaluate_lab_models(models_dict, X_train_clean, y_train_clean, X_test, y_test)

Evaluating models on ORIGINAL data...
Evaluating models on CLEANED data...


In [14]:
print("\n--- Results: Original Data ---")
display(results_original.sort_values(by="F1-score", ascending=False))

print("\n--- Results: Cleaned Data ---")
display(results_cleaned.sort_values(by="F1-score", ascending=False))


--- Results: Original Data ---


,Model name,Accuracy,Precision,Recall,F1-score
0,Naive Bayes,0.853,0.8530,0.853,0.8530
1,Random Forest,0.834,0.8343,0.834,0.8340
2,RBF SVM,0.829,0.8290,0.829,0.8290
3,KNN,0.772,0.7731,0.772,0.7718
4,Neural Net,0.757,0.7570,0.757,0.7570
5,Linear SVM,0.720,0.7204,0.720,0.7199
6,AdaBoost,0.719,0.7665,0.719,0.7059
7,Decision Tree,0.671,0.7337,0.671,0.6473



--- Results: Cleaned Data ---


,Model name,Accuracy,Precision,Recall,F1-score
0,RBF SVM,0.971,0.9710,0.971,0.9710
1,Linear SVM,0.967,0.9670,0.967,0.9670
2,Naive Bayes,0.956,0.9561,0.956,0.9560
3,Random Forest,0.941,0.9414,0.941,0.9410
4,Neural Net,0.940,0.9403,0.940,0.9400
5,KNN,0.916,0.9199,0.916,0.9158
6,Decision Tree,0.905,0.9086,0.905,0.9048
7,AdaBoost,0.897,0.9019,0.897,0.8967
